# Getting Started with Stable Diffusion

Run all cells in a fresh Python3.11 kernel. The original SDv1-4 checkpoint is public and pinned; no token is required. Full original19 end-to-end images and the100-step manual image are generated, not small placeholder models. CUDA, Apple MPS or CPU is selected; CPU can take substantial time. Four CPU threads bound fallback use. Safety checking stays enabled, including the manual final image. The final independent LMS comparison adds a second100-step denoising run. Original27 complete immutable gists remain in the appendix. Output cells/counts are cleared before committing.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from tempfile import TemporaryDirectory
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', 'torch==2.10.0', 'torchvision==0.25.0', 'transformers==4.57.6', 'diffusers==0.36.0', 'accelerate==1.12.0', 'scipy==1.16.3', 'ftfy==6.3.1', 'numpy==2.3.5', 'pillow==12.0.0', 'certifi==2026.10.23', 'ipywidgets==8.1.7'], check=True)
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
import numpy as np
import torch
import diffusers
from PIL import Image
from IPython.display import display
import hashlib
import time
torch.set_num_threads(4)
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
MODEL_ID = 'CompVis/stable-diffusion-v1-4'
MODEL_REVISION = '133a221b8aa7292a167afc5127cb63fb5005638b'
workspace = TemporaryDirectory(prefix='tdb-stable-diffusion-')
work_dir = Path(workspace.name)
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| Diffusers:', diffusers.__version__)
print('Device:', DEVICE, '| dtype:', DTYPE)


In [ ]:
print('CUDA available:', torch.cuda.is_available())
print('Apple MPS available:', torch.backends.mps.is_available())
print('Selected execution device:', DEVICE)

In [ ]:
print('Pinned Diffusers, Transformers, SciPy, ftfy and notebook widgets installed in the active kernel.')

In [ ]:
import ipywidgets
print('Notebook widgets:', ipywidgets.__version__)

In [ ]:
from huggingface_hub import HfApi
model_info = HfApi(token=False).model_info(MODEL_ID, revision=MODEL_REVISION)
assert model_info.sha == MODEL_REVISION and model_info.gated is False
print('Public checkpoint:', MODEL_ID, '| pinned revision:', model_info.sha)
print('Authentication required for this checkpoint:', model_info.gated)

In [ ]:
from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, variant='fp16', torch_dtype=DTYPE,
    use_safetensors=True, token=False,
)
pipe.set_progress_bar_config(disable=True)
pipe.vae.enable_slicing()
if DEVICE == 'mps':
    pipe.enable_attention_slicing()
assert pipe.safety_checker is not None
print('Loaded original Stable Diffusion v1-4 components and enabled safety checker.')

In [ ]:
pipe = pipe.to(DEVICE)
generated_images, run_records = [], []
def generate_images(prompts, steps=50, seed=None, height=512, width=512):
    # A CPU generator works with CUDA, MPS and CPU through Diffusers' noise helper.
    generator = torch.Generator('cpu').manual_seed(seed) if seed is not None else None
    records = []
    def record_step(pipeline, step_index, timestep, callback_kwargs):
        latents = callback_kwargs['latents']
        assert torch.isfinite(latents).all().item()
        records.append((step_index, float(timestep.detach().cpu())))
        return callback_kwargs
    count = 1 if isinstance(prompts, str) else len(prompts)
    # MPS batch inference can exceed unified memory; generate every requested
    # image sequentially, preserving the full prompt count and denoising steps.
    prompt_batches = list(prompts) if DEVICE == 'mps' and not isinstance(prompts,str) else [prompts]
    images,flags = [],[]
    for batch in prompt_batches:
        result = pipe(batch, num_inference_steps=steps, generator=generator,
                      height=height, width=width, callback_on_step_end=record_step)
        images.extend(result.images)
        flags.extend(result.nsfw_content_detected)
    assert len(images) == count
    run_records.append(dict(count=count, requested_steps=steps, completed_steps=len(records),
                            seed=seed, size=(width,height), safety_flags=flags))
    generated_images.extend(images)
    print('Generated:', count, '| size:', (width,height), '| denoising steps:', len(records), '| safety flags:', flags)
    return images
print('Pipeline moved to:', DEVICE, '| generation recorder ready.')

In [ ]:
prompt = 'a photograph of an astronaut riding a horse'
image = generate_images(prompt)[0]
image.save(work_dir / 'astronaut_rides_horse.png')
display(image)

In [ ]:
generator = torch.Generator('cpu').manual_seed(1024)
image = generate_images(prompt, seed=1024)[0]
seeded_image = image.copy()
display(image)

In [ ]:
generator = torch.Generator('cpu').manual_seed(1024)
image = generate_images(prompt, steps=15, seed=1024)[0]
display(image)

In [ ]:
def image_grid(imgs, rows, cols):
    if rows <= 0 or cols <= 0 or len(imgs) != rows * cols:
        raise ValueError('The positive grid dimensions must match the image count.')
    w, h = imgs[0].size
    if any(img.size != (w,h) for img in imgs):
        raise ValueError('Grid images must have the same dimensions.')
    grid = Image.new('RGB', (cols*w, rows*h))
    for index, img in enumerate(imgs):
        grid.paste(img, ((index%cols)*w, (index//cols)*h))
    return grid
print('Image-grid helper ready.')

In [ ]:
num_images = 3
prompt = ['a photograph of an astronaut riding a horse'] * num_images
images = generate_images(prompt)
grid = image_grid(images, rows=1, cols=3)
display(grid)
print('Three-image grid:', grid.size)

In [ ]:
num_cols, num_rows = 3, 4
prompt = ['a photograph of an astronaut riding a horse'] * num_cols
all_images = []
for row in range(num_rows):
    images = generate_images(prompt)
    all_images.extend(images)
grid = image_grid(all_images, rows=num_rows, cols=num_cols)
display(grid)
print('Twelve-image grid:', grid.size)

In [ ]:
prompt = 'a photograph of an astronaut riding a horse'
image = generate_images(prompt, height=512, width=768)[0]
display(image)

In [ ]:
torch_device = DEVICE
print('Manual pipeline device:', torch_device)

In [ ]:
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, UNet2DConditionModel
# Reuse the exact already-loaded components to avoid a second copy of the weights.
vae, tokenizer, text_encoder, unet = pipe.vae, pipe.tokenizer, pipe.text_encoder, pipe.unet
print('Components:', type(vae).__name__, type(tokenizer).__name__, type(text_encoder).__name__, type(unet).__name__)
print('Checkpoint latent scaling factor:', vae.config.scaling_factor)

In [ ]:
from diffusers import LMSDiscreteScheduler
scheduler = LMSDiscreteScheduler(beta_start=0.00085, beta_end=0.012,
                                 beta_schedule='scaled_linear', num_train_timesteps=1000)
print(type(scheduler).__name__)

In [ ]:
vae, text_encoder, unet = vae.to(torch_device), text_encoder.to(torch_device), unet.to(torch_device)
print('All manual components use:', torch_device)

In [ ]:
prompt = ['a photograph of an astronaut riding a horse']
height, width = 512, 512
num_inference_steps = 100
guidance_scale = 7.5
generator = torch.Generator('cpu').manual_seed(32)
batch_size = 1
print('Manual generation:', num_inference_steps, 'steps | seed: 32 | size:', (width,height))

In [ ]:
text_input = tokenizer(prompt, padding='max_length', max_length=tokenizer.model_max_length,
                       truncation=True, return_tensors='pt')
with torch.inference_mode():
    text_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]
print('Conditional text embeddings:', tuple(text_embeddings.shape))

In [ ]:
max_length = text_input.input_ids.shape[-1]
uncond_input = tokenizer([''] * batch_size, padding='max_length', max_length=max_length, return_tensors='pt')
with torch.inference_mode():
    uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]
print('Unconditional text embeddings:', tuple(uncond_embeddings.shape))

In [ ]:
text_embeddings = torch.cat([uncond_embeddings, text_embeddings])
print('Classifier-free guidance embedding batch:', tuple(text_embeddings.shape))

In [ ]:
latents = torch.randn((batch_size, unet.config.in_channels, height//8, width//8), generator=generator)
initial_noise = latents.clone()
latents = latents.to(device=torch_device, dtype=DTYPE)
print('Initial latent shape:', tuple(latents.shape))

In [ ]:
scheduler.set_timesteps(num_inference_steps)
print('Actual LMS timesteps:', len(scheduler.timesteps))

In [ ]:
latents = latents * scheduler.init_noise_sigma
print('Initial noise sigma:', round(float(scheduler.init_noise_sigma), 6))

In [ ]:
manual_steps = []
for step_index, timestep in enumerate(scheduler.timesteps):
    latent_model_input = scheduler.scale_model_input(torch.cat([latents] * 2), timestep)
    with torch.inference_mode():
        noise_pred = unet(latent_model_input, timestep, encoder_hidden_states=text_embeddings).sample
    noise_uncond, noise_text = noise_pred.chunk(2)
    guided_noise = noise_uncond + guidance_scale * (noise_text - noise_uncond)
    latents = scheduler.step(guided_noise, timestep, latents).prev_sample
    assert torch.isfinite(latents).all().item()
    manual_steps.append(step_index)
manual_final_latents = latents.detach().clone()
print('Manual denoising steps completed:', len(manual_steps))
print('Final latent shape:', tuple(latents.shape))

In [ ]:
scaled_latents = latents / vae.config.scaling_factor
with torch.inference_mode():
    image = vae.decode(scaled_latents).sample
manual_decoded = image.detach().clone()
print('Decoded image tensor:', tuple(image.shape))

In [ ]:
image_array = (image / 2 + 0.5).clamp(0, 1).detach().cpu().permute(0,2,3,1).numpy()
# Apply the same safety checker as the end-to-end pipeline before displaying images.
checked_images, manual_safety = pipe.run_safety_checker(image_array, torch_device, DTYPE)
pil_images = pipe.image_processor.numpy_to_pil(checked_images)
display(pil_images[0])
print('Manual image:', pil_images[0].size, '| safety flags:', manual_safety)

In [ ]:
assert MODEL_REVISION == model_info.sha
assert len(generated_images) == 19
assert [record['count'] for record in run_records] == [1,1,1,3,3,3,3,3,1]
assert [record['requested_steps'] for record in run_records] == [50,50,15,50,50,50,50,50,50]
assert all(record['completed_steps'] >= record['requested_steps'] for record in run_records)
assert all(image.size == (512,512) for image in generated_images[:-1])
assert generated_images[-1].size == (768,512)
assert len(manual_steps) == 100 and manual_steps == list(range(100))
assert tuple(text_embeddings.shape) == (2,77,768)
assert tuple(manual_final_latents.shape) == (1,4,64,64)
assert tuple(manual_decoded.shape) == (1,3,512,512)
assert pipe.safety_checker is not None and len(manual_safety) == 1
for record in run_records:
    assert len(record['safety_flags']) == record['count']
    assert all(isinstance(flag,(bool,np.bool_))for flag in record['safety_flags'])
# Compare the full manual100-step result to Diffusers' separate LMS pipeline path.
original_scheduler = pipe.scheduler
pipe.scheduler = LMSDiscreteScheduler(beta_start=0.00085,beta_end=0.012,beta_schedule='scaled_linear',num_train_timesteps=1000)
reference = pipe(prompt, num_inference_steps=100, guidance_scale=7.5,
                 latents=initial_noise.clone(), output_type='latent').images
torch.testing.assert_close(manual_final_latents, reference, rtol=2e-4, atol=2e-4)
pipe.scheduler = original_scheduler
# Grid placement is checked with independently colored tiles, not generated-image similarity.
tiles = [Image.new('RGB',(4,3),color) for color in [(255,0,0),(0,255,0),(0,0,255),(255,255,0)]]
tile_grid = np.asarray(image_grid(tiles,2,2))
for index,tile in enumerate(tiles):
    x,y=(index%2)*4,(index//2)*3
    np.testing.assert_array_equal(tile_grid[y:y+3,x:x+4],np.asarray(tile))
for bad in [([],1,1),(tiles,0,4),(tiles,1,3)]:
    try:image_grid(*bad)
    except ValueError:pass
    else:raise AssertionError('Invalid grid must fail.')
# Pixel and latent element ratio includes channels: 3*512*512/(4*64*64)=48.
assert (3*512*512)//(4*64*64) == 48
assert torch.isfinite(manual_decoded).all().item()
print('All19 full pipeline images, requested50/15-step runs,3- and12-image grids and768×512 output completed.')
print('Full100-step manual LMS latents match independent Diffusers LMS inference; all shapes, safety checks, grid placement and48× element ratio passed.')
workspace.cleanup()
print('Owned temporary image files cleaned up.')

## Original immutable source gists

Nonexecuting originals, preserved verbatim:

### Original block 1

Gist 1f8e65d0bfad1050e8605d8cfb3881d4 revision 231eabe93978511d1c036c01283ebaf1f2f3ef93

```text
!nvidia-smi
```

### Original block 2

Gist 9e8e69e08d2c11bf7a880af5d54e37dd revision cc503857f55fbcb06e45e35798f828ed7c734645

```text
!pip install diffusers==0.4.0
!pip install transformers scipy ftfy
!pip install "ipywidgets>=7,<8"
```

### Original block 3

Gist e0249c624d3158a002c76699b9e41495 revision 8d478144d1ebdd8d6e9849ef584dea440c59d71d

```text
from google.colab import output
output.enable_custom_widget_manager()
```

### Original block 4

Gist c7e70cf6d6380db9e4ac76ab97ebd195 revision 33193d0e39644ee497e852754bdc185653885a97

```text
from huggingface_hub import notebook_login

notebook_login()
```

### Original block 5

Gist 6bc77b802bd955e1e852f4dec02ab940 revision 579f9059c43f98afe3a6efa60083f9b5eb7ab1c3

```text
import torch
from diffusers import StableDiffusionPipeline

# make sure you're logged in with `huggingface-cli login`
pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4", revision="fp16", torch_dtype=torch.float16)  
```

### Original block 6

Gist 13014bdea29193bb1b915afca0746453 revision 39060df891a5d9973c8333cd82855a2d4ad91668

```text
pipe = pipe.to("cuda")
```

### Original block 7

Gist 517bcc8f3d172593cb29116b39d7708d revision aa7a21773a095a178b58620819ed5fd05ddd1214

```text
prompt = "a photograph of an astronaut riding a horse"
image = pipe(prompt).images[0]  # image here is in [PIL format](https://pillow.readthedocs.io/en/stable/)

# Now to display an image you can do either save it such as:
image.save(f"astronaut_rides_horse.png")
```

### Original block 8

Gist 83971b339893ba681dd0b77974714941 revision 2026c095135264fb592981818297d00e4d0a4f0a

```text
import torch

generator = torch.Generator("cuda").manual_seed(1024)

image = pipe(prompt, generator=generator).images[0]

image
```

### Original block 9

Gist b058a65185e0b73a682a300323390079 revision d1c8b4aba792d89603ea4b54aa2a3f72b0322b41

```text
import torch

generator = torch.Generator("cuda").manual_seed(1024)

image = pipe(prompt, num_inference_steps=15, generator=generator).images[0]

image
```

### Original block 10

Gist 46da60457917d8a06c389d86299a718f revision cb28abf94c1ee8093f29297de6256de98c98529f

```text
from PIL import Image

def image_grid(imgs, rows, cols):
    assert len(imgs) == rows*cols

    w, h = imgs[0].size
    grid = Image.new('RGB', size=(cols*w, rows*h))
    grid_w, grid_h = grid.size
    
    for i, img in enumerate(imgs):
        grid.paste(img, box=(i%cols*w, i//cols*h))
    return grid
```

### Original block 11

Gist 6b418c703bb1778171c0d517db2859cf revision b0c5629fb367b36639258bc980bc68af9095afb6

```text
num_images = 3
prompt = ["a photograph of an astronaut riding a horse"] * num_images

images = pipe(prompt).images

grid = image_grid(images, rows=1, cols=3)
grid
```

### Original block 12

Gist cd42882f7913bc1d8fc701ca8125a9af revision 0f7c4827a11e5e93883c8a79d64e144bee0e2a6a

```text
num_cols = 3
num_rows = 4

prompt = ["a photograph of an astronaut riding a horse"] * num_cols

all_images = []
for i in range(num_rows):
  images = pipe(prompt).images
  all_images.extend(images)

grid = image_grid(all_images, rows=num_rows, cols=num_cols)
grid
```

### Original block 13

Gist e57d39b09be4743f93e61dd3ea0c2318 revision 52154c1ce0238547b5dbb8f391bb1ba0a24df056

```text
prompt = "a photograph of an astronaut riding a horse"

image = pipe(prompt, height=512, width=768).images[0]
image
```

### Original block 14

Gist 6e8ceef4b19891e9f656e94c938a75a2 revision a0e8d7df80bc9f4fca19045816abfc7e8884d7bd

```text
import torch
torch_device = "cuda" if torch.cuda.is_available() else "cpu"
```

### Original block 15

Gist 42a66022c7a50b8c1a9cbefb2a471d33 revision e3e85beb87dec3767e3e892998a53463049f8aff

```text
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, UNet2DConditionModel, PNDMScheduler

# 1. Load the autoencoder model which will be used to decode the latents into image space. 
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae")

# 2. Load the tokenizer and text encoder to tokenize and encode the text. 
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")

# 3. The UNet model for generating the latents.
unet = UNet2DConditionModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="unet")
```

### Original block 16

Gist 32cca0795896a1c23c16a5daf90716b1 revision 9fe101c425cf72e08bbf7209216c0e6c5743d276

```text
from diffusers import LMSDiscreteScheduler

scheduler = LMSDiscreteScheduler(beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear", num_train_timesteps=1000)
```

### Original block 17

Gist 09e440d61390bb6567b8c5e96b6e4bb8 revision 9190ef120fc740f6ec2aed19d08b92ff01dee638

```text
vae = vae.to(torch_device)
text_encoder = text_encoder.to(torch_device)
unet = unet.to(torch_device) 
```

### Original block 18

Gist 39c25247050ebe1ffe04129b974c6761 revision f30b483ca1edc2e59eb18abaf052e64e863eff7b

```text
prompt = ["a photograph of an astronaut riding a horse"]

height = 512                        # default height of Stable Diffusion
width = 512                         # default width of Stable Diffusion

num_inference_steps = 100            # Number of denoising steps

guidance_scale = 7.5                # Scale for classifier-free guidance

generator = torch.manual_seed(32)   # Seed generator to create the inital latent noise

batch_size = 1
```

### Original block 19

Gist cd7175738787d56d1cedc9eb4b720446 revision d21ffe63c74ca422a4816595f0f08dfeb0eff791

```text
text_input = tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")

with torch.no_grad():
  text_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]
```

### Original block 20

Gist 19fb25ffd7b0779268d8c7517475265f revision 8e3418758eef58e37acd9eff092a875910c03bed

```text
max_length = text_input.input_ids.shape[-1]
uncond_input = tokenizer(
    [""] * batch_size, padding="max_length", max_length=max_length, return_tensors="pt"
)
with torch.no_grad():
  uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]   
```

### Original block 21

Gist f8f86f18bbdcac4854a6449d9cb62713 revision 08367f28e7ec6de0c8281b5b570181e9c3e49479

```text
text_embeddings = torch.cat([uncond_embeddings, text_embeddings])
```

### Original block 22

Gist d40c286de8dc46cfcbac6abeb710d296 revision d46738630b578e2d2abb0d75565861b826efea3d

```text
latents = torch.randn(
  (batch_size, unet.in_channels, height // 8, width // 8),
  generator=generator,
)
latents = latents.to(torch_device)
```

### Original block 23

Gist eed76aa49f2ed3f2d5c03354f00ee025 revision 28afc544bb0cf27158aa4c6f1e3037e0681750cc

```text
scheduler.set_timesteps(num_inference_steps)
```

### Original block 24

Gist e447d52b189d9a15c5e67197803f8447 revision eadbbe9467fdf412ece6a026d89c6115dd9ed77a

```text
latents = latents * scheduler.init_noise_sigma
```

### Original block 25

Gist 59ce66d8cd3ab70f695a90c8c69e78f8 revision 0952665f4f7c72c308f8cc131b59bcbe954b49c9

```text
from tqdm.auto import tqdm
from torch import autocast

for t in tqdm(scheduler.timesteps):
  # expand the latents if we are doing classifier-free guidance to avoid doing two forward passes.
  latent_model_input = torch.cat([latents] * 2)

  latent_model_input = scheduler.scale_model_input(latent_model_input, t)

  # predict the noise residual
  with torch.no_grad():
    noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings).sample

  # perform guidance
  noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
  noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

  # compute the previous noisy sample x_t -> x_t-1
  latents = scheduler.step(noise_pred, t, latents).prev_sample
```

### Original block 26

Gist 7bf844ab0bedcd764b7999f8e7dd28fe revision cbf42a19a7df1f44beefe6d3f6415152876d907c

```text
# scale and decode the image latents with vae
latents = 1 / 0.18215 * latents

with torch.no_grad():
  image = vae.decode(latents).sample
```

### Original block 27

Gist b442f4fa26e740fd43453453ba2fadf1 revision ff8cbf5a6cb4873f221232ba49aaf5b6095bc691

```text
image = (image / 2 + 0.5).clamp(0, 1)
image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
images = (image * 255).round().astype("uint8")
pil_images = [Image.fromarray(image) for image in images]
pil_images[0]
```